# Campaign Analysis

Vestal Pro launch. Need to deterimine

Need to determine:
- Incremental revenue
- Cannibalization
- Halo effect
- Channel efficiency -- were they efficient
- Any opportunity cost -- ie stockouts or revenue left on the table

## Summary

- footwear revenue up **+35.8%** vs baseline during the campaign = **+$69.3K**
- total net revenue **+33.1%** vs baseline = **+$77.5K**
- Vestal Pro itself did **$99.8K** direct revenue
- HOWEVER, ~**$30.5K** of that appears to have shifted from other shoes (cannibalization. more below)
- apparel/accessories picked up ~**$8.2K** (halo)
- incremental media spend (above normal run-rate) was **$43.0K**, so net new revenue after that lands around **$34.5K** → **1.80×** return on the incremental spend
- Hit revenue goal, hit new cx goal, hit halo goal
- cannibalization is the issue. Estimates land **19% to 32%** of Vestal Pro units. Need to monitor and watch. Ultra Glide's getting hit hardest.
- inventory held during 20-day campaign but core sizes ran out afterward
- returns + CS contacts also ticked up post-campaign, mostly sizing

## Analysis Structure

- phases come from `campaign_phase`: pre-campaign Mar 1-May 31 (92 days), campaign Jun 1-Jun 20 (20 days), post Jun 21-Jul 20 (30 days)
- baseline forecast = pre-campaign trend + day-of-week seasonality. tested on last 2 weeks of pre-period before trusting it on the campaign window
- CIs from bootstrapping the residuals against bsaeline forecast
- net new revenue checked two ways -- total site revenue vs baseline, and Vestal Pro direct + halo - cannibalization. should roughly agree. they do, within rounding
- cannibalization uses 3 baselines (flat avg, last-4-weeks avg, trend model) instead of trusting one number. shown as a range
- no CRM holdout or geo test, so channel iROAS = attributed ROAS × the same site-level incrementality factor for every channel. directional only, not a real channel test
- "incremental media" = spend above the normal 20-day run rate, not total spend

### Assumptions
- returns could still creep up, last few weeks are still fresh
- no COGS in the data so ROI numbers are revenue-based not profit. Margin assumed at 60% for VP sales. 
- cannibalization limit + other goals are in README.md

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 20)

# Palette and typography sampled from the campaign readout deck.
SALOMON_BLUE = "#0057B8"
BLUE_BRIGHT = "#3E8EDE"
BLUE_LIGHT = "#BFD5ED"
SALOMON_GREEN = "#1F7A4D"
INK = "#333333"
MUTED = "#757575"
GRID = "#D9D9D9"
SURFACE = "#F5F6F7"
WHITE = "#FFFFFF"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PLOTS_DIR = PROJECT_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
PNG_WIDTH = 1400
PNG_SCALE = 2

salomon_template = go.layout.Template(
    layout=go.Layout(
        font=dict(family="Arial, Helvetica, sans-serif", size=12, color=INK),
        title=dict(font=dict(family="Georgia, Times New Roman, serif", size=20, color=INK), x=0.01),
        colorway=[SALOMON_BLUE, BLUE_BRIGHT, SALOMON_GREEN, BLUE_LIGHT, MUTED],
        paper_bgcolor=WHITE,
        plot_bgcolor=WHITE,
        hoverlabel=dict(bgcolor=WHITE, bordercolor=GRID, font=dict(family="Arial", color=INK)),
        legend=dict(font=dict(color=INK), bgcolor="rgba(255,255,255,0)"),
        xaxis=dict(
            showline=True, linecolor=GRID, linewidth=1, mirror=False,
            gridcolor=GRID, gridwidth=0.7, zerolinecolor=GRID,
            tickfont=dict(color=MUTED), title_font=dict(color=INK),
        ),
        yaxis=dict(
            showline=True, linecolor=GRID, linewidth=1, mirror=False,
            gridcolor=GRID, gridwidth=0.7, zerolinecolor=GRID,
            tickfont=dict(color=MUTED), title_font=dict(color=INK),
        ),
    )
)
pio.templates["salomon_readout"] = salomon_template
pio.templates.default = "salomon_readout"
pio.renderers.default = "plotly_mimetype+notebook_connected"

PLOTLY_CONFIG = {
    "displaylogo": False,
    "responsive": True,
    "modeBarButtonsToRemove": ["lasso2d", "select2d"],
    "toImageButtonOptions": {"format": "png", "filename": "salomon_campaign_chart", "scale": 2},
}


def show_campaign_chart(
    fig, title, subtitle=None, *, height=420, hovermode="closest", left_margin=64,
    width=None, export_name=None,
):
    """Apply the shared readout styling and display an interactive Plotly figure."""
    fig.update_layout(
        title=dict(
            text=title,
            subtitle=dict(
                text=subtitle or "",
                font=dict(family="Arial, Helvetica, sans-serif", size=12, color=MUTED),
            ),
            font=dict(family="Georgia, Times New Roman, serif", size=21, color=INK),
            x=0, xref="paper", xanchor="left",
            y=0.98, yref="container", yanchor="top",
            automargin=True, pad=dict(b=12),
        ),
        width=width,
        height=height + 24,
        margin=dict(l=left_margin, r=36, t=116 if subtitle else 92, b=58),
        hovermode=hovermode,
        bargap=0.36,
    )
    fig.update_xaxes(showline=True, linecolor=GRID, gridcolor=GRID, ticks="outside", tickcolor=GRID)
    fig.update_yaxes(showline=True, linecolor=GRID, gridcolor=GRID, ticks="outside", tickcolor=GRID)
    if export_name:
        output_path = PLOTS_DIR / export_name
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Support for Kaleido versions less than 1.0.0")
            fig.write_image(
                output_path, width=width or PNG_WIDTH, height=height + 24, scale=PNG_SCALE, format="png"
            )
        print(f"Saved {output_path.relative_to(PROJECT_ROOT)}")
    fig.show(config=PLOTLY_CONFIG)


# Light notebook chrome so headings and tables echo the deck as well as the charts.
display(HTML("""
<style>
.jp-RenderedHTMLCommon, .text_cell_render { font-family: Arial, Helvetica, sans-serif; color: #333333; }
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3,
.text_cell_render h1, .text_cell_render h2, .text_cell_render h3 {
    font-family: Georgia, 'Times New Roman', serif; color: #333333;
}
.jp-RenderedHTMLCommon h1, .text_cell_render h1 { border-top: 6px solid #0057B8; padding-top: 14px; }
.jp-RenderedHTMLCommon h2, .text_cell_render h2 { border-bottom: 2px solid #BFD5ED; padding-bottom: 5px; }
.dataframe { border-collapse: collapse; font-family: Arial, Helvetica, sans-serif; }
.dataframe thead th { background: #0057B8 !important; color: white !important; border: 1px solid #0057B8 !important; }
.dataframe tbody th { color: #333333; background: #BFD5ED; border: 1px solid #D9D9D9; }
.dataframe tbody td { border: 1px solid #D9D9D9; }
.dataframe tbody tr:nth-child(even) td { background: #F5F6F7; }
</style>
"""))

PRE_START = pd.Timestamp("2026-03-01")
PRE_END = pd.Timestamp("2026-05-31")
ACT_START = pd.Timestamp("2026-06-01")
ACT_END = pd.Timestamp("2026-06-20")
POST_END = pd.Timestamp("2026-07-20")
ACTIVE_DAYS = 20

EXISTING_FRANCHISES = ["Ultra Glide", "Genesis", "Speedcross"]
ADJACENT = ["Trail Apparel", "Trail Accessories"]
PAID_CHANNELS = ["Email", "SMS", "Paid Social", "Paid Search", "Affiliate"]

RNG = np.random.default_rng(7)
N_BOOT = 4000

## Data used

Same mock CSVs as notebook 02:
- `ecommerce_daily.csv` -- daily site rev + adjacent-category rev
- `product_daily.csv` -- units/revenue/availability per shoe
- `channel_daily.csv` -- spend + attributed revenue per channel
- `customer_orders.csv` -- customer type, opt-ins, returns

Charts are Plotly, same colors as the deck. Also saves PNGs to `plots/` when run.

In [2]:
ROOT = Path.cwd()
if not (ROOT / "data" / "mock").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "mock"

ecommerce = pd.read_csv(DATA_DIR / "ecommerce_daily.csv", parse_dates=["date"])
products = pd.read_csv(DATA_DIR / "product_daily.csv", parse_dates=["date"])
channels = pd.read_csv(DATA_DIR / "channel_daily.csv", parse_dates=["date"])
orders = pd.read_csv(DATA_DIR / "customer_orders.csv", parse_dates=["order_date"])
services = pd.read_csv(DATA_DIR / "consumer_services.csv", parse_dates=["contact_date"])

for name, frame, col in [
    ("ecommerce_daily", ecommerce, "date"), ("product_daily", products, "date"),
    ("channel_daily", channels, "date"), ("customer_orders", orders, "order_date"),
]:
    print(f"{name}: {len(frame):,} rows, {frame[col].min():%b %d} – {frame[col].max():%b %d}")

ecommerce_daily: 142 rows, Mar 01 – Jul 20
product_daily: 852 rows, Mar 01 – Jul 20
channel_daily: 994 rows, Mar 01 – Jul 20
customer_orders: 13,652 rows, Mar 01 – Jul 20


In [3]:
ecommerce.head(5)

,date,campaign_phase,sessions,orders,units,gross_revenue,net_revenue,conversion_rate,average_order_value,footwear_revenue,adjacent_category_revenue,return_rate
0,2026-03-01,Pre-campaign,3607,82,92,11915.71,10075.96,0.0227,122.88,8733.27,1342.71,0.1196
1,2026-03-02,Pre-campaign,3604,84,91,11252.59,10554.98,0.0233,125.65,9121.04,1433.94,0.0440
2,2026-03-03,Pre-campaign,3850,91,110,13662.61,11928.51,0.0236,131.08,9524.42,2404.09,0.0909
3,2026-03-04,Pre-campaign,3347,77,96,10720.95,9827.86,0.0230,127.63,7489.64,2338.20,0.0521
4,2026-03-05,Pre-campaign,3257,77,84,10570.46,9769.28,0.0236,126.87,8235.80,1533.48,0.0476


In [4]:
products.head(5)

,date,campaign_phase,franchise,category,sessions,units_sold,net_revenue,average_selling_price,inventory_ats,in_stock_rate,core_size_availability,return_rate
0,2026-03-01,Pre-campaign,Vestal Pro,Footwear,0,0,0.00,0.00,0,0.0000,0.0000,0.0000
1,2026-03-01,Pre-campaign,Genesis,Footwear,613,26,3319.50,127.67,2774,0.9666,0.9468,0.0769
2,2026-03-01,Pre-campaign,Ultra Glide,Footwear,540,22,2772.92,126.04,2378,0.9170,0.8954,0.2273
3,2026-03-01,Pre-campaign,Speedcross,Footwear,416,19,2640.85,138.99,2681,0.9786,0.9623,0.0526
4,2026-03-01,Pre-campaign,Trail Apparel,Adjacent,251,15,907.62,60.51,4385,0.9616,0.9301,0.2000


In [5]:
# Focused input checks before leaning on anything.
assert ecommerce["date"].is_unique
assert not products.duplicated(["date", "franchise"]).any()
assert not channels.duplicated(["date", "channel"]).any()
assert orders["order_id"].is_unique
assert set(ecommerce["campaign_phase"]) == {"Pre-campaign", "Active campaign", "Post-campaign"}
assert (ecommerce["date"].min(), ecommerce["date"].max()) == (PRE_START, POST_END)
assert products[["units_sold", "net_revenue", "core_size_availability"]].notna().all().all()
assert (channels["spend"] >= 0).all()
print("Input checks passed: unique keys, expected phases and date range, no nulls in the fields used below.")

Input checks passed: unique keys, expected phases and date range, no nulls in the fields used below.


## Results

### Revenue vs baseline

- used the 92 pre-campaign days to fit trend + day-of-week
- trained on first 78 days, tested on last 2 weeks before trusting it on the campaign period

In [6]:
def expected_baseline(frame, ycol, fit_through=PRE_END): 
    """Fit trend + day-of-week on rows up to fit_through; return frame with an
    'expected' column over all rows, plus pre-period residuals."""
    frame = frame.sort_values("date").reset_index(drop=True)
    X = pd.get_dummies(frame["date"].dt.dayofweek, prefix="dow", drop_first=True).astype(float) # day of week seasonality
    X.insert(0, "t", (frame["date"] - frame["date"].min()).dt.days) # trend
    X.insert(0, "const", 1.0) # constant
    fit = frame["date"] <= fit_through # period to fit the model on
    beta, *_ = np.linalg.lstsq(X[fit].to_numpy(), frame.loc[fit, ycol].to_numpy(), rcond=None) # fit the model using OLS
    out = frame.copy()
    out["expected"] = X.to_numpy() @ beta # beta is coeffs multipled by X (constant and trend + day-of-week)
    resid = out.loc[fit, ycol].to_numpy() - out.loc[fit, "expected"].to_numpy() 
    return out, resid

# Backtest on the last 14 pre-period days.
pre_only = ecommerce[ecommerce["date"] <= PRE_END]
backtest, _ = expected_baseline(pre_only, "footwear_revenue", fit_through=PRE_END - pd.Timedelta(days=14))
holdout = backtest[backtest["date"] > PRE_END - pd.Timedelta(days=14)]
mape = (holdout["footwear_revenue"] - holdout["expected"]).abs().div(holdout["footwear_revenue"]).mean()
print(f"The two-week test period had a MAPE of {mape:.1%}. This is accurate enough for the 20-day campaign comparison.")

The two-week test period had a MAPE of 8.0%. This is accurate enough for the 20-day campaign comparison.


In [7]:
footwear, footwear_resid = expected_baseline(ecommerce, "footwear_revenue")
active_fw = footwear[footwear["campaign_phase"] == "Active campaign"]

observed = active_fw["footwear_revenue"].sum()
expected = active_fw["expected"].sum()
incremental_fw = observed - expected

boot = np.array([
    (active_fw["footwear_revenue"].to_numpy()
     - active_fw["expected"].to_numpy()
     - RNG.choice(footwear_resid, len(active_fw), replace=True)).sum()
    for _ in range(N_BOOT)
])
ci_low, ci_high = np.percentile(boot, [2.5, 97.5]) / expected

# Repeat the calculation for total net revenue.
net, _ = expected_baseline(ecommerce, "net_revenue")
active_net = net[net["campaign_phase"] == "Active campaign"]
incremental_net = active_net["net_revenue"].sum() - active_net["expected"].sum()

print(f"Active footwear revenue: ${observed/1e3:,.1f}K observed vs ${expected/1e3:,.1f}K expected")
print(f"Incremental footwear revenue: +${incremental_fw/1e3:,.1f}K (+{incremental_fw/expected:.1%}), "
      f"95% CI {ci_low:+.1%} to {ci_high:+.1%}")
print(f"Incremental total net revenue: +${incremental_net/1e3:,.1f}K "
      f"(+{incremental_net/active_net['expected'].sum():.1%})")

Active footwear revenue: $262.8K observed vs $193.5K expected
Incremental footwear revenue: +$69.3K (+35.8%), 95% CI +32.0% to +39.7%
Incremental total net revenue: +$77.5K (+33.1%)


In [8]:
# FILTER PLOT DATA TO ONLY INCLUDE APRIL 15-END
START = pd.Timestamp("2026-04-15")
footwear = footwear[footwear["date"] >= START]

fig = go.Figure()

# Shade only the modeled incremental gap during the active flight.
fig.add_trace(go.Scatter(
    x=active_fw["date"], y=active_fw["expected"],
    mode="lines", line=dict(width=0), hoverinfo="skip", showlegend=False,
))
fig.add_trace(go.Scatter(
    x=active_fw["date"], y=active_fw["footwear_revenue"],
    mode="lines", line=dict(width=0), fill="tonexty",
    fillcolor="rgba(0, 87, 184, 0.16)", hoverinfo="skip", showlegend=False,
))
fig.add_trace(go.Scatter(
    x=footwear["date"], y=footwear["footwear_revenue"],
    name="Observed", mode="lines",
    line=dict(color=SALOMON_BLUE, width=2.4),
    hovertemplate="%{x|%b %d}<br>Observed: $%{y:,.0f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=footwear["date"], y=footwear["expected"],
    name="Modeled no-campaign baseline", mode="lines",
    line=dict(color=MUTED, width=2, dash="dash"),
    hovertemplate="%{x|%b %d}<br>Baseline: $%{y:,.0f}<extra></extra>",
))
fig.add_vline(x=ACT_START, line=dict(color=MUTED, width=1))
fig.add_vline(x=ACT_END, line=dict(color=MUTED, width=1))
fig.add_annotation(
    x=ACT_START + (ACT_END - ACT_START) / 2, y=0.97, xref="x", yref="paper",
    text="ACTIVE FLIGHT", showarrow=False, yanchor="top",
    bgcolor="rgba(255,255,255,0.82)", borderpad=3, font=dict(size=10, color=MUTED),
)
fig.update_layout(
    legend=dict(
        orientation="v", x=0.01, xanchor="left", y=0.98, yanchor="top",
        bgcolor="rgba(255,255,255,0.86)", bordercolor=GRID, borderwidth=1,
    ),
)
fig.update_xaxes(title=None)
fig.update_yaxes(title="Daily footwear revenue", tickprefix="$", tickformat="~s")
show_campaign_chart(
    fig,
    "Footwear revenue vs modeled baseline",
    height=600,
    width=1100,
    hovermode="x unified",
    export_name="01_footwear_revenue_vs_baseline.png",
)

/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/01_footwear_revenue_vs_baseline.png


The above is for all footwear revenue. 

### Where the incremental revenue came from

Breakdown:
- direct Vestal Pro revenue
- sales shifted from existing shoes (cannibalization)
- extra apparel/accessories revenue (halo, positive)

each piece gets its own baseline. add them up and check against the site-level number above.

In [9]:
vestal = products[products["franchise"] == "Vestal Pro"]
vestal_active = vestal[vestal["campaign_phase"] == "Active campaign"]
vestal_direct = vestal_active["net_revenue"].sum()
vestal_units = vestal_active["units_sold"].sum()

# Franchise shift in revenue, trend + day-of-week baseline per franchise.
shift_rows = []
for franchise in EXISTING_FRANCHISES:
    frame, _ = expected_baseline(products[products["franchise"] == franchise], "net_revenue")
    active = frame[frame["campaign_phase"] == "Active campaign"]
    shift_rows.append({"Franchise": franchise,
                       "Observed": active["net_revenue"].sum(),
                       "Expected": active["expected"].sum()})
shift = pd.DataFrame(shift_rows)
shift["Delta"] = shift["Observed"] - shift["Expected"]
franchise_shift = -shift["Delta"].sum()  # positive = revenue displaced

# Additional apparel and accessory revenue during the campaign.
adjacent, adjacent_resid = expected_baseline(ecommerce, "adjacent_category_revenue")
adj_active = adjacent[adjacent["campaign_phase"] == "Active campaign"]
halo_revenue = adj_active["adjacent_category_revenue"].sum() - adj_active["expected"].sum()

bottom_up = vestal_direct - franchise_shift + halo_revenue

ledger = pd.DataFrame({
    "Component": ["Vestal Pro direct revenue", "Sales shifted from existing shoes",
                  "Additional apparel and accessories", "Combined product estimate",
                  "Site-level estimate"],
    "Value": [vestal_direct, -franchise_shift, halo_revenue, bottom_up, incremental_net],
})
ledger["Value"] = ledger["Value"].map(lambda v: f"${v/1e3:+,.1f}K")
display(ledger)

assert abs(bottom_up - incremental_net) < 3_000
print(f"The product-level estimate is ${bottom_up/1e3:,.1f}K and the site-level estimate is "
      f"${incremental_net/1e3:,.1f}K, a difference of ${abs(bottom_up - incremental_net)/1e3:,.1f}K.")

,Component,Value
0,Vestal Pro direct revenue,$+99.8K
1,Sales shifted from existing shoes,$-30.5K
2,Additional apparel and accessories,$+8.2K
3,Combined product estimate,$+77.5K
4,Site-level estimate,$+77.5K


The product-level estimate is $77.5K and the site-level estimate is $77.5K, a difference of $0.0K.


In [10]:
# Count only media spend above the normal pre-campaign run rate.
paid = channels[channels["channel"].isin(PAID_CHANNELS)]
active_spend = paid[paid["campaign_phase"] == "Active campaign"]["spend"].sum()
pre_runrate = paid[paid["campaign_phase"] == "Pre-campaign"]["spend"].sum() / 92 * ACTIVE_DAYS
incremental_media = active_spend - pre_runrate
net_new = incremental_net - incremental_media

print(f"Campaign paid spend was ${active_spend/1e3:,.1f}K, compared with a normal 20-day run rate of ${pre_runrate/1e3:,.1f}K")
print(f"Additional media spend was ${incremental_media/1e3:,.1f}K, leaving ${net_new/1e3:,.1f}K after media")
print(f"Return on incremental media: {incremental_net/incremental_media:.2f}x (revenue basis; no COGS in this data)")

Campaign paid spend was $74.6K, compared with a normal 20-day run rate of $31.6K
Additional media spend was $43.0K, leaving $34.5K after media
Return on incremental media: 1.80x (revenue basis; no COGS in this data)


### Net campaign impact

Waterfall of the pieces above:
- direct Vestal Pro revenue
- minus cannibalization
- plus halo
- = net incremental revenue
- minus incremental media spend = net campaign value
- minus COGS = net contribution

Assuming **60% gross margin** for the last step. Assumption. red = cost

In [11]:
MARGIN_RATE = 0.60  # assumed contribution margin; the mock data carries no COGS
SALOMON_RED = "#DA291C"  # deck secondary red, used by the Sankey below

net_value = bottom_up - incremental_media
cogs = (1 - MARGIN_RATE) * bottom_up
contribution = net_value - cogs

ledger_rows = [
    ("Vestal Pro direct revenue", vestal_direct, "gain"),
    ("Cannibalization", -franchise_shift, "cost"),
    ("Adjacent-category halo", halo_revenue, "gain"),
    ("Net incremental revenue", bottom_up, "total"),
    ("Direct Marketing Spend", -incremental_media, "cost"),
    ("Net campaign value (revenue basis)", net_value, "total"),
    ("COGS", -cogs, "cost"),
    ("Contribution Margin", contribution, "total"),
]

fill = {"gain": SALOMON_BLUE, "cost": WHITE, "total": "#000000"}
edge = {"gain": SALOMON_BLUE, "cost": "#000000", "total": "#000000"}
values = [row[1] for row in ledger_rows]

fig = go.Figure(go.Bar(
    x=values,
    y=[row[0] for row in ledger_rows],
    orientation="h",
    marker=dict(
        color=[fill[row[2]] for row in ledger_rows],
        line=dict(color=[edge[row[2]] for row in ledger_rows], width=1.1),
    ),
    text=[f"\u2212${abs(v)/1e3:,.0f}K" if v < 0 else f"${v/1e3:,.0f}K" for v in values],
    textposition="outside",
    textfont=dict(color=INK, size=12),
    cliponaxis=False,
    hovertemplate="%{y}<br>$%{x:,.0f}<extra></extra>",
))
fig.add_vline(x=0, line=dict(color=INK, width=1))
fig.update_yaxes(autorange="reversed", showgrid=False, title=None)
fig.update_xaxes(visible=False, showgrid=False, range=[-62_000, 116_000])
show_campaign_chart(
    fig,
    "Net campaign impact",
    height=560,
    width=1000,
    left_margin=232,
    export_name="06_net_campaign_impact.png",
)

print(f"Net incremental revenue: ${bottom_up/1e3:,.1f}K")
print(f"Net campaign value after ${incremental_media/1e3:,.1f}K incremental media: ${net_value/1e3:,.1f}K")
print(f"Net contribution at a {MARGIN_RATE:.0%} margin: ${contribution/1e3:,.1f}K")


Saved plots/06_net_campaign_impact.png


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Net incremental revenue: $77.5K
Net campaign value after $43.0K incremental media: $34.5K
Net contribution at a 60% margin: $3.5K


Visualize the flow chart in a different way... TESTING:
- Vestal Pro direct + halo → gross campaign revenue
- splits into cannibalization, COGS, incremental media, and net contribution

note: cannibalization pulled out at full revenue not net. works because COGS is only charged against the net incremental piece. COGS on displaced units would've happened either way.

In [12]:
gross_revenue = vestal_direct + halo_revenue
cogs_flow = (1 - MARGIN_RATE) * bottom_up

sankey_nodes = [
    f"Vestal Pro direct revenue \u00b7 ${vestal_direct/1e3:,.1f}K",
    f"Apparel & accessories halo \u00b7 ${halo_revenue/1e3:,.1f}K",
    f"Gross campaign revenue \u00b7 ${gross_revenue/1e3:,.1f}K",
    f"Offset by cannibalization \u00b7 ${franchise_shift/1e3:,.1f}K",
    f"COGS ({1 - MARGIN_RATE:.0%} of net incremental) \u00b7 ${cogs_flow/1e3:,.1f}K",
    f"Incremental paid media \u00b7 ${incremental_media/1e3:,.1f}K",
    f"Net contribution margin \u00b7 ${contribution/1e3:,.1f}K",
]
node_colors = [SALOMON_BLUE, BLUE_BRIGHT, "#000000",
               SALOMON_RED, SALOMON_RED, SALOMON_RED, SALOMON_GREEN]
BLUE_TINT = "rgba(0, 87, 184, 0.30)"
RED_TINT = "rgba(218, 41, 28, 0.28)"
GREEN_TINT = "rgba(31, 122, 77, 0.55)"

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        label=sankey_nodes, color=node_colors,
        pad=30, thickness=14, line=dict(width=0),
        hovertemplate="%{label}<extra></extra>",
    ),
    link=dict(
        source=[0, 1, 2, 2, 2, 2],
        target=[2, 2, 3, 4, 5, 6],
        value=[vestal_direct, halo_revenue,
               franchise_shift, cogs_flow, incremental_media, contribution],
        color=[BLUE_TINT, BLUE_TINT, RED_TINT, RED_TINT, RED_TINT, GREEN_TINT],
        hovertemplate="%{source.label} \u2192 %{target.label}<br>$%{value:,.0f}<extra></extra>",
    ),
))
show_campaign_chart(
    fig,
    "Where each campaign dollar went",
    height=440,
    export_name="07_net_campaign_impact_sankey.png",
)

print(f"Gross campaign-attributable revenue: ${gross_revenue/1e3:,.1f}K")
print(f"Outflows \u2014 cannibalization ${franchise_shift/1e3:,.1f}K, COGS ${cogs_flow/1e3:,.1f}K, "
      f"media ${incremental_media/1e3:,.1f}K \u2014 leave ${contribution/1e3:,.1f}K of contribution margin")


Saved plots/07_net_campaign_impact_sankey.png


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Gross campaign-attributable revenue: $108.1K
Outflows — cannibalization $30.5K, COGS $31.0K, media $43.0K — leave $3.5K of contribution margin


Not as goood at the waterfall above. 

### Sales shifted from existing shoes

Units first, then revenue.
- 3 baselines per shoe, shown as a range not one number
- Ultra Glide takes the biggest hit, Genesis next
- Speedcross ticks up a bit, offsets some of the shift

In [13]:
def unit_baselines(franchise):
    frame = products[products["franchise"] == franchise]
    pre = frame[frame["campaign_phase"] == "Pre-campaign"]
    active_units = frame[frame["campaign_phase"] == "Active campaign"]["units_sold"].sum()
    trend_frame, _ = expected_baseline(frame, "units_sold")
    return {
        "observed": active_units,
        "flat": pre["units_sold"].mean() * ACTIVE_DAYS,
        "recent": pre[pre["date"] > PRE_END - pd.Timedelta(days=28)]["units_sold"].mean() * ACTIVE_DAYS,
        "trend": trend_frame[trend_frame["campaign_phase"] == "Active campaign"]["expected"].sum(),
    }

specs = {fr: unit_baselines(fr) for fr in EXISTING_FRANCHISES}
spec_names = ["flat", "recent", "trend"]
unit_change_by_spec = {
    spec: {fr: specs[fr]["observed"] / specs[fr][spec] - 1 for fr in EXISTING_FRANCHISES}
    for spec in spec_names
}
assert all(min(changes, key=changes.get) == "Ultra Glide" for changes in unit_change_by_spec.values())

detail = pd.DataFrame({
    "Franchise": EXISTING_FRANCHISES,
    "Active units": [specs[fr]["observed"] for fr in EXISTING_FRANCHISES],
}).set_index("Franchise")
for spec in spec_names:
    detail[f"vs {spec}"] = [f"{specs[fr]['observed'] / specs[fr][spec] - 1:+.1%}" for fr in EXISTING_FRANCHISES]
display(detail)

rates = {}
for spec in spec_names:
    displaced = sum(max(specs[fr][spec] - specs[fr]["observed"], 0) for fr in ["Genesis", "Ultra Glide"])
    gained = max(specs["Speedcross"]["observed"] - specs["Speedcross"][spec], 0)
    rates[spec] = (displaced - gained) / vestal_units
    print(f"{spec:>6} estimate: {displaced - gained:>4.0f} units shifted, equal to {rates[spec]:.1%} of Vestal Pro units")

rate_range = (min(rates.values()), max(rates.values()))
print(f"\nEstimated sales shifted from existing shoes: {rate_range[0]:.0%}–{rate_range[1]:.0%} of Vestal Pro units. "
      f"The median estimate is {np.median(list(rates.values())):.0%}, compared with the goal of staying below 10%. "
      f"All three estimates are above that goal.")

,Active units,vs flat,vs recent,vs trend
Franchise,,,,
Ultra Glide,323,-22.8%,-26.7%,-28.5%
Genesis,453,-13.7%,-14.5%,-17.5%
Speedcross,444,+7.8%,+0.6%,-1.0%


  flat estimate:  135 units shifted, equal to 19.4% of Vestal Pro units
recent estimate:  192 units shifted, equal to 27.7% of Vestal Pro units
 trend estimate:  225 units shifted, equal to 32.4% of Vestal Pro units

Estimated sales shifted from existing shoes: 19%–32% of Vestal Pro units. The median estimate is 28%, compared with the goal of staying below 10%. All three estimates are above that goal.


In [14]:
central = {fr: np.mean([specs[fr]["observed"] / specs[fr][s] - 1 for s in spec_names])
           for fr in EXISTING_FRANCHISES}
spans = {fr: [specs[fr]["observed"] / specs[fr][s] - 1 for s in spec_names] for fr in EXISTING_FRANCHISES}

values = np.array([central[fr] for fr in EXISTING_FRANCHISES])
range_low = np.array([min(spans[fr]) for fr in EXISTING_FRANCHISES])
range_high = np.array([max(spans[fr]) for fr in EXISTING_FRANCHISES])

fig = go.Figure(go.Bar(
    x=values,
    y=EXISTING_FRANCHISES,
    orientation="h",
    marker=dict(
        color=[SALOMON_BLUE if franchise == "Ultra Glide" else BLUE_LIGHT if central[franchise] < 0 else MUTED
               for franchise in EXISTING_FRANCHISES],
        line=dict(color=INK, width=0.7),
    ),
    error_x=dict(
        type="data", symmetric=False,
        array=range_high - values, arrayminus=values - range_low,
        color=INK, thickness=1.3, width=5,
    ),
    text=[f"{value:+.0%}" for value in values],
    textposition="outside",
    textfont=dict(color=INK, size=12),
    cliponaxis=False,
    customdata=np.column_stack([range_low, range_high]),
    hovertemplate=(
        "%{y}<br>Mean vs baseline: %{x:+.1%}"
        "<br>Specification range: %{customdata[0]:+.1%} to %{customdata[1]:+.1%}<extra></extra>"
    ),
))
fig.add_vline(x=0, line=dict(color=INK, width=1))
fig.update_xaxes(
    title="Campaign units vs expected baseline",
    tickformat="+.0%",
    range=[range_low.min() - 0.09, range_high.max() + 0.09],
)
fig.update_yaxes(autorange="reversed", title=None, showgrid=False)
show_campaign_chart(
    fig,
    "Existing-franchise units vs expected baseline",
    height=370,
    left_margin=112,
    export_name="02_existing_franchise_units_vs_baseline.png",
)

Saved plots/02_existing_franchise_units_vs_baseline.png


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Can use this in the deck for cannibalization. Shows the range of estimates, and which shoes are most affected.

In [15]:
deck_vals = [central[fr] for fr in EXISTING_FRANCHISES]

fig = go.Figure(go.Bar(
    x=EXISTING_FRANCHISES,
    y=deck_vals,
    marker=dict(
        color=[WHITE if v < 0 else SALOMON_GREEN for v in deck_vals],
        line=dict(color=[INK if v < 0 else SALOMON_GREEN for v in deck_vals], width=1.1),
    ),
    customdata=[[min(spans[fr]), max(spans[fr])] for fr in EXISTING_FRANCHISES],
    hovertemplate=("%{x}<br>Central estimate: %{y:+.1%}"
                   "<br>Range across baselines: %{customdata[0]:+.1%} to %{customdata[1]:+.1%}<extra></extra>"),
))

# Dashed whiskers spanning the three baseline specifications, with solid caps.
CAP_HALF_WIDTH = 0.09
for i, fr in enumerate(EXISTING_FRANCHISES):
    lo, hi = min(spans[fr]), max(spans[fr])
    fig.add_shape(type="line", x0=i, x1=i, y0=lo, y1=hi,
                  line=dict(color=INK, width=1.3, dash="dash"))
    for y in (lo, hi):
        fig.add_shape(type="line", x0=i - CAP_HALF_WIDTH, x1=i + CAP_HALF_WIDTH, y0=y, y1=y,
                      line=dict(color=INK, width=1.3))
    value = central[fr]
    below = value < 0
    fig.add_annotation(
        x=i, y=lo if below else hi, text=f"<b>{value:+.0%}</b>", showarrow=False,
        yanchor="top" if below else "bottom", yshift=-6 if below else 6,
        font=dict(color=INK, size=13),
    )

fig.add_hline(y=0, line=dict(color=INK, width=1))
fig.update_xaxes(title=None, showgrid=False, tickfont=dict(color=MUTED, size=13))
fig.update_yaxes(visible=False, showgrid=False, range=[-0.36, 0.14])
show_campaign_chart(
    fig,
    "Campaign-period unit change vs expected baseline",
    height=510,
    width=900,
    export_name="08_cannibalization_by_franchise.png",
)


Saved plots/08_cannibalization_by_franchise.png


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Actually, I like this chart more. Easier to read. 

### Revenue displaced from existing shoes

Same 3-baseline approach on revenue instead of units:
- range is **\$18.7K to \$30.5K**
- recent 4-week avg is **\$25.9K**, right in the middle
- using the trend estimate (**\$30.5K**) for the ledger and the deck. it's the one that reconciles with the site-level number. anything lower and the two ways of counting this stop agreeing.

In [16]:
# Revenue counterpart to the unit ranges above, so the single figure used in the
def revenue_displaced(spec):
    total = 0.0
    for franchise in EXISTING_FRANCHISES:
        frame = products[products["franchise"] == franchise]
        pre = frame[frame["campaign_phase"] == "Pre-campaign"]
        observed = frame[frame["campaign_phase"] == "Active campaign"]["net_revenue"].sum()
        if spec == "flat":
            expected = pre["net_revenue"].mean() * ACTIVE_DAYS
        elif spec == "recent":
            expected = pre[pre["date"] > PRE_END - pd.Timedelta(days=28)]["net_revenue"].mean() * ACTIVE_DAYS
        else:
            trend_frame, _ = expected_baseline(frame, "net_revenue")
            expected = trend_frame[trend_frame["campaign_phase"] == "Active campaign"]["expected"].sum()
        total += expected - observed  # positive = revenue displaced
    return total

displaced_by_spec = {spec: revenue_displaced(spec) for spec in spec_names}
for spec in spec_names:
    print(f"{spec:>6} baseline: ${displaced_by_spec[spec]/1e3:5.1f}K displaced "
          f"({displaced_by_spec[spec]/vestal_direct:.0%} of Vestal Pro direct revenue)")

# The trend spec here is the franchise_shift used in the ledger; keep them tied.
assert abs(displaced_by_spec["trend"] - franchise_shift) < 1.0

low, high = min(displaced_by_spec.values()), max(displaced_by_spec.values())
print(f"\nRevenue displaced from existing shoes: ${low/1e3:,.1f}K\u2013${high/1e3:,.1f}K across baselines. "
      f"I carry the trend estimate of ${franchise_shift/1e3:,.1f}K because it reconciles the product-level "
      f"total (${bottom_up/1e3:,.1f}K) with the site-level estimate (${incremental_net/1e3:,.1f}K).")


  flat baseline: $ 18.7K displaced (19% of Vestal Pro direct revenue)
recent baseline: $ 25.9K displaced (26% of Vestal Pro direct revenue)
 trend baseline: $ 30.5K displaced (31% of Vestal Pro direct revenue)

Revenue displaced from existing shoes: $18.7K–$30.5K across baselines. I carry the trend estimate of $30.5K because it reconciles the product-level total ($77.5K) with the site-level estimate ($77.5K).


### Apparel and accessories

Checking apparel/accessories vs baseline, and whether the bump is attached to Vestal Pro orders or separate purchases.

In [17]:
halo_rows = []
for label, frame, ycol in [
    ("Trail Apparel", products[products["franchise"] == "Trail Apparel"], "net_revenue"),
    ("Trail Accessories", products[products["franchise"] == "Trail Accessories"], "net_revenue"),
    ("All adjacent", ecommerce, "adjacent_category_revenue"),
]:
    fitted, resid = expected_baseline(frame, ycol)
    active = fitted[fitted["campaign_phase"] == "Active campaign"]
    obs, exp = active[ycol].sum(), active["expected"].sum()
    draws = np.array([
        (active[ycol].to_numpy() - active["expected"].to_numpy()
         - RNG.choice(resid, len(active), replace=True)).sum() / exp
        for _ in range(N_BOOT)
    ])
    halo_rows.append({"Category": label, "Lift": obs / exp - 1,
                      "CI low": np.percentile(draws, 2.5), "CI high": np.percentile(draws, 97.5)})
halo = pd.DataFrame(halo_rows)
halo_display = halo.copy()
for col in ["Lift", "CI low", "CI high"]:
    halo_display[col] = halo_display[col].map("{:+.1%}".format)
display(halo_display)

# Check whether the increase came from attached items or separate orders.
active_orders = orders[orders["campaign_phase"] == "Active campaign"]
vestal_orders = active_orders[active_orders["primary_product"] == "Vestal Pro"]
other_fw = active_orders[active_orders["primary_product"].isin(EXISTING_FRANCHISES)]
standalone_pre = orders[(orders["campaign_phase"] == "Pre-campaign")
                        & (orders["primary_product"].isin(ADJACENT))]["total_order_revenue"].sum() / 92
standalone_act = active_orders[active_orders["primary_product"].isin(ADJACENT)]["total_order_revenue"].sum() / ACTIVE_DAYS

print(f"Attach rate: {(vestal_orders['adjacent_product_revenue'] > 0).mean():.1%} of Vestal Pro orders "
      f"vs {(other_fw['adjacent_product_revenue'] > 0).mean():.1%} of other footwear orders")
print(f"Apparel and accessory revenue attached to Vestal Pro orders: "
      f"${vestal_orders['adjacent_product_revenue'].sum()/1e3:,.1f}K, close to the total modeled increase "
      f"of ${halo_revenue/1e3:,.1f}K")
print(f"Standalone adjacent demand: ${standalone_pre:,.0f}/day pre vs ${standalone_act:,.0f}/day active "
      f"({standalone_act/standalone_pre - 1:+.0%}) — flat")

,Category,Lift,CI low,CI high
0,Trail Apparel,+17.4%,+7.8%,+26.4%
1,Trail Accessories,+26.5%,+14.2%,+38.9%
2,All adjacent,+20.3%,+13.6%,+27.0%


Attach rate: 21.6% of Vestal Pro orders vs 17.5% of other footwear orders
Apparel and accessory revenue attached to Vestal Pro orders: $9.3K, close to the total modeled increase of $8.2K
Standalone adjacent demand: $1,306/day pre vs $1,323/day active (+1%) — flat


In [18]:
fig = go.Figure(go.Bar(
    x=halo["Lift"],
    y=halo["Category"],
    orientation="h",
    marker=dict(color=SALOMON_BLUE, line=dict(color=INK, width=0.7)),
    error_x=dict(
        type="data", symmetric=False,
        array=halo["CI high"] - halo["Lift"],
        arrayminus=halo["Lift"] - halo["CI low"],
        color=INK, thickness=1.3, width=5,
    ),
    text=halo["Lift"].map("{:+.0%}".format),
    textposition="outside",
    textfont=dict(color=INK, size=12),
    cliponaxis=False,
    customdata=halo[["CI low", "CI high"]],
    hovertemplate=(
        "%{y}<br>Lift: %{x:+.1%}"
        "<br>95% CI: %{customdata[0]:+.1%} to %{customdata[1]:+.1%}<extra></extra>"
    ),
))
fig.add_vline(x=0.05, line=dict(color=SALOMON_GREEN, width=1.5, dash="dash"))
fig.add_annotation(
    x=0.05, y=0.96, xref="x", yref="paper", text="+5% TARGET",
    showarrow=False, xanchor="left", yanchor="top",
    bgcolor="rgba(255,255,255,0.82)", borderpad=3, font=dict(size=10, color=SALOMON_GREEN),
)
fig.update_xaxes(
    title="Campaign revenue vs expected baseline",
    tickformat="+.0%",
    range=[0, halo["CI high"].max() + 0.08],
)
fig.update_yaxes(autorange="reversed", title=None, showgrid=False)
show_campaign_chart(
    fig,
    "Apparel and accessory revenue vs baseline",
    height=360,
    left_margin=142,
    export_name="03_adjacent_category_revenue_lift.png",
)

Saved plots/03_adjacent_category_revenue_lift.png


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Deck version of the halo chart. pass criterion called out, apparel/accessories in blue, combined number in black.

In [19]:
# Deck-formatted adjacent-category lift for the halo slide.
halo_colors = [SALOMON_BLUE if cat != "All adjacent" else "#000000" for cat in halo["Category"]]

fig = go.Figure(go.Bar(
    x=halo["Category"],
    y=halo["Lift"],
    marker=dict(color=halo_colors, line=dict(color=INK, width=0.7)),
    text=[f"{v:+.1%}" for v in halo["Lift"]],
    textposition="outside",
    textfont=dict(color=INK, size=13),
    cliponaxis=False,
    customdata=halo[["CI low", "CI high"]],
    hovertemplate=("%{x}<br>Lift: %{y:+.1%}"
                   "<br>95% CI: %{customdata[0]:+.1%} to %{customdata[1]:+.1%}<extra></extra>"),
))
fig.add_hline(y=0, line=dict(color=INK, width=1))

all_row = halo[halo["Category"] == "All adjacent"].iloc[0]
fig.add_annotation(
    x=0.0, y=-0.14, xref="paper", yref="paper", xanchor="left", yanchor="top", align="left",
    text=(f"Success criterion \u2265 +5%: <b style='color:{SALOMON_GREEN}'>PASS</b>  "
          f"({all_row['Lift']:+.1%}, 95% CI {all_row['CI low']:+.1%} to {all_row['CI high']:+.1%} "
          f"\u00b7 +${halo_revenue/1e3:,.1f}K)"),
    font=dict(size=11, color=MUTED), showarrow=False,
)
fig.update_xaxes(title=None, showgrid=False, tickfont=dict(color=INK, size=13))
fig.update_yaxes(visible=False, range=[0, halo["Lift"].max() * 1.25])
show_campaign_chart(
    fig, "Adjacent-category revenue vs expected baseline",
    height=430, width=760, export_name="12_adjacent_category_halo.png",
)


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/12_adjacent_category_halo.png


This is the halo chart that I will use in the deck

### New customers

Goal: ≥25% of Vestal Pro buyers new to Salomon. Also checking acquisition cost, early value, repeat purchase rate, email/SMS sign-ups.

In [20]:
vestal_buyer_type = (vestal_orders.sort_values(["order_date", "order_id"])
                   .groupby("customer_id")["customer_type"].first())
new_ids = set(active_orders.loc[active_orders["customer_type"] == "New", "customer_id"])
first_orders = (active_orders[active_orders["customer_type"] == "New"]
                .sort_values(["order_date", "order_id"])
                .groupby("customer_id")["total_order_revenue"].first())
early_value = orders[orders["customer_id"].isin(new_ids)].groupby("customer_id")["total_order_revenue"].sum()
post_orders = orders[orders["campaign_phase"] == "Post-campaign"]
repeat_rate = post_orders[post_orders["customer_id"].isin(new_ids)]["customer_id"].nunique() / len(new_ids)
blended_cac = active_spend / len(new_ids)

pre_orders = orders[orders["campaign_phase"] == "Pre-campaign"]
adds = {
    "Email": (pre_orders["email_opt_in"].sum() / 92, active_orders["email_opt_in"].sum() / ACTIVE_DAYS),
    "SMS": (pre_orders["sms_opt_in"].sum() / 92, active_orders["sms_opt_in"].sum() / ACTIVE_DAYS),
}

acquisition = pd.DataFrame({
    "Metric": [
        "Vestal Pro active buyers", "— of which new to Salomon", "New-customer share",
        "New customers, all active orders", "Blended CAC (active paid spend / new customers)",
        "Average first order, new customers", "Early customer value (through Jul 20)",
        "Early value / CAC", "Post-window repeat rate",
    ],
    "Value": [
        f"{len(vestal_buyer_type):,}", f"{int((vestal_buyer_type == 'New').sum()):,}",
        f"{(vestal_buyer_type == 'New').mean():.1%}", f"{len(new_ids):,}",
        f"${blended_cac:,.0f}", f"${first_orders.mean():,.0f}", f"${early_value.mean():,.0f}",
        f"{early_value.mean() / blended_cac:.1f}x", f"{repeat_rate:.1%}",
    ],
})
display(acquisition)

for channel_name, (pre_rate, act_rate) in adds.items():
    print(f"{channel_name} adds/day: {pre_rate:.1f} pre -> {act_rate:.1f} active ({act_rate/pre_rate-1:+.0%})")
new_active = active_orders[active_orders["customer_type"] == "New"]
existing_active = active_orders[active_orders["customer_type"] == "Existing"]
print(f"Opt-in rates, active orders: new customers {new_active['email_opt_in'].mean():.0%} email / "
      f"{new_active['sms_opt_in'].mean():.0%} SMS vs existing {existing_active['email_opt_in'].mean():.0%} / "
      f"{existing_active['sms_opt_in'].mean():.0%}")
print(f"New customers drove ${new_active['total_order_revenue'].sum()/1e3:,.1f}K "
      f"({new_active['total_order_revenue'].sum()/active_orders['total_order_revenue'].sum():.1%} of active net revenue)")

,Metric,Value
0,Vestal Pro active buyers,667
1,— of which new to Salomon,268
2,New-customer share,40.2%
3,"New customers, all active orders",730
4,Blended CAC (active paid spend / new customers),$102
5,"Average first order, new customers",$129
6,Early customer value (through Jul 20),$188
7,Early value / CAC,1.8x
8,Post-window repeat rate,28.1%


Email adds/day: 50.3 pre -> 68.0 active (+35%)
SMS adds/day: 24.7 pre -> 33.9 active (+37%)


Opt-in rates, active orders: new customers 72% email / 40% SMS vs existing 51% / 24%
New customers drove $94.2K (30.2% of active net revenue)


Deck versions -- buyer mix + database growth, matches the acquisition slide.

In [21]:
# Buyer mix donut for the acquisition slide.
new_ct = int((vestal_buyer_type == "New").sum())
existing_ct = int((vestal_buyer_type == "Existing").sum())
new_share = new_ct / len(vestal_buyer_type)

fig = go.Figure(go.Pie(
    labels=[f"New to Salomon \u00b7 {new_ct}", f"Existing \u00b7 {existing_ct}"],
    values=[new_ct, existing_ct],
    hole=0.62, sort=False, direction="clockwise",
    marker=dict(colors=[SALOMON_BLUE, GRID], line=dict(color=WHITE, width=2)),
    textinfo="none",
    hovertemplate="%{label}: %{percent}<extra></extra>",
))
fig.add_annotation(text=f"<b>{new_share:.0%}</b>", x=0.5, y=0.54, showarrow=False,
                   font=dict(size=36, color=INK))
fig.add_annotation(text="new", x=0.5, y=0.40, showarrow=False, font=dict(size=15, color=MUTED))
goal_pass = "PASS" if new_share >= 0.25 else "MISS"
fig.add_annotation(
    x=0.99, y=0.99, xref="paper", yref="paper", xanchor="right", yanchor="top",
    text=f"\u2265 25% NEW \u00b7 <b>{goal_pass}</b>", showarrow=False,
    bgcolor="rgba(255,255,255,0.85)", borderpad=4,
    font=dict(size=11, color=SALOMON_GREEN if goal_pass == "PASS" else "#B3261E"),
)
fig.update_layout(legend=dict(orientation="h", y=-0.06, x=0.5, xanchor="center", font=dict(color=INK)))
show_campaign_chart(
    fig, "Vestal Pro buyer mix",
    height=470, width=560, export_name="09_vestal_pro_buyer_mix.png",
)


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/09_vestal_pro_buyer_mix.png


In [22]:
# Database opt-in adds per day, pre-period vs active campaign.
channel_labels = ["Email", "SMS"]
pre_vals = [adds[c][0] for c in channel_labels]
act_vals = [adds[c][1] for c in channel_labels]

fig = go.Figure()
fig.add_bar(name="Pre-period", x=channel_labels, y=pre_vals, marker_color=GRID,
            text=[f"{v:.1f}" for v in pre_vals], textposition="outside",
            textfont=dict(color=MUTED, size=12))
fig.add_bar(name="Active campaign", x=channel_labels, y=act_vals, marker_color=SALOMON_BLUE,
            text=[f"{v:.1f}" for v in act_vals], textposition="outside",
            textfont=dict(color=INK, size=12))
for i, c in enumerate(channel_labels):
    fig.add_annotation(x=c, y=act_vals[i], yshift=30, showarrow=False,
                       text=f"<b>{act_vals[i]/pre_vals[i]-1:+.0%}</b>",
                       font=dict(size=12, color=SALOMON_BLUE))
fig.update_layout(barmode="group", bargap=0.42, bargroupgap=0.12,
                  legend=dict(orientation="h", y=-0.08, x=0.5, xanchor="center", font=dict(color=INK)))
fig.update_yaxes(visible=False, range=[0, max(act_vals) * 1.28])
fig.update_xaxes(title=None, tickfont=dict(color=INK, size=13))
show_campaign_chart(
    fig, "Database adds per day",
    height=470, width=640, export_name="10_database_adds_per_day.png",
)


Saved plots/10_database_adds_per_day.png


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


### Projected LTV of new customers

Can't measure real LTV yet, not enough history. this is a projection.
- Borrow from earlier work: assuming **3.0×** first-order value over 24 months (net of returns), front-loaded
- avg first order **$129** → 24-month value ~**$387**/customer → **3.8×** the **$102** blended CAC
- already collected **$188** through July 20. covers CAC inside month 1, ahead of the assumed curve. so this looks conservative
- ~**86%** of projected value lands in year 1
- 268 new-to-Salomon VP buyers → ~**$104K** projected 24-month value, or ~**$283K** extended to all 730 new customers across all products during campaign

In [23]:
# Projected cumulative value per new customer as a multiple of the first order.
# CLV_TARGET_MULT / CLV_TAU are planning assumptions from prior cohort analysis;
# replace with measured cohort curves once post-campaign history matures.
CLV_TARGET_MULT = 3.0   # first-order value returned by month 24, net of returns
CLV_HORIZON_M = 24
CLV_TAU = 9.0           # months; shape of the cumulative-value curve

first_order_new = first_orders.mean()
observed_value = early_value.mean()
month = np.arange(0, CLV_HORIZON_M + 1)
mult = 1 + (CLV_TARGET_MULT - 1) * (1 - np.exp(-month / CLV_TAU)) / (1 - np.exp(-CLV_HORIZON_M / CLV_TAU))
clv = first_order_new * mult
clv_24m = clv[-1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=month, y=clv, mode="lines", line=dict(color=SALOMON_BLUE, width=2.6),
    name="Projected cumulative value",
    hovertemplate="Month %{x}<br>$%{y:,.0f}<extra></extra>",
))
fig.add_hline(y=blended_cac, line=dict(color=MUTED, width=1.5, dash="dash"))
fig.add_annotation(x=CLV_HORIZON_M, y=blended_cac, yshift=-12, xanchor="right", showarrow=False,
                   text=f"Blended CAC ${blended_cac:,.0f}", font=dict(size=11, color=MUTED))
fig.add_trace(go.Scatter(
    x=[0], y=[first_order_new], mode="markers+text", showlegend=False,
    marker=dict(color=SALOMON_BLUE, size=10),
    text=[f"First order ${first_order_new:,.0f}"], textposition="top right",
    textfont=dict(size=11, color=INK),
))
fig.add_trace(go.Scatter(
    x=[1.3], y=[observed_value], mode="markers+text", showlegend=False,
    marker=dict(color=SALOMON_GREEN, size=11, symbol="diamond"),
    text=[f"  Observed to date ${observed_value:,.0f}"], textposition="middle right",
    textfont=dict(size=11, color=SALOMON_GREEN),
))
fig.add_annotation(
    x=CLV_HORIZON_M, y=clv_24m, yshift=14, xanchor="right", showarrow=False,
    text=f"<b>${clv_24m:,.0f}</b> \u00b7 {CLV_TARGET_MULT:.1f}\u00d7 first order",
    font=dict(size=12, color=SALOMON_BLUE),
)
fig.update_layout(legend=dict(
    x=0.03, y=0.97, xanchor="left", yanchor="top",
    bgcolor="rgba(255,255,255,0.85)", bordercolor=GRID, borderwidth=1,
    font=dict(color=INK),
))
fig.update_xaxes(title="Months since first purchase", dtick=6, range=[-0.6, CLV_HORIZON_M + 0.6])
fig.update_yaxes(title="Cumulative value per new customer", tickprefix="$", rangemode="tozero")
show_campaign_chart(
    fig, "Projected new-customer lifetime value",
    height=460, width=780, export_name="11_new_customer_clv.png",
)

print(f"Assumed {CLV_TARGET_MULT:.1f}x first-order value over {CLV_HORIZON_M} months "
      f"(prior cohort analysis).")
print(f"Projected 24-month value: ${clv_24m:,.0f} per new customer vs ${blended_cac:,.0f} CAC "
      f"= {clv_24m/blended_cac:.1f}x LTV:CAC.")
print(f"Across {new_ct} new Vestal Pro buyers: ${new_ct*clv_24m/1e3:,.0f}K projected 24-month value.")
print(f"Value is front-loaded: ${clv[12]:,.0f} by month 12 = {clv[12]/clv_24m:.0%} of the 24-month total.")
print(f"Extended to all {len(new_ids)} new customers in the flight: "
      f"${len(new_ids)*clv_24m/1e3:,.0f}K projected 24-month value.")


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/11_new_customer_clv.png


Assumed 3.0x first-order value over 24 months (prior cohort analysis).
Projected 24-month value: $387 per new customer vs $102 CAC = 3.8x LTV:CAC.
Across 268 new Vestal Pro buyers: $104K projected 24-month value.
Value is front-loaded: $333 by month 12 = 86% of the 24-month total.
Extended to all 730 new customers in the flight: $283K projected 24-month value.


### Channel performance

Table shows attributed ROAS first, then applies the same site-level incrementality factor across every channel. no holdouts, so this is an estimate.
- email/SMS probably getting too much credit. high-intent audience already
- paid social probably getting too little credit. likely doing upper-funnel work that shows up elsewhere

In [24]:
active_channels = channels[channels["campaign_phase"] == "Active campaign"]
paid_summary = (active_channels[active_channels["channel"].isin(PAID_CHANNELS)]
                .groupby("channel")
                .agg(spend=("spend", "sum"), attributed=("attributed_revenue", "sum"),
                     new_customers=("new_customers", "sum")))
incrementality_factor = incremental_net / paid_summary["attributed"].sum()
paid_summary["roas"] = paid_summary["attributed"] / paid_summary["spend"]
paid_summary["iroas"] = paid_summary["roas"] * incrementality_factor
paid_summary["cac_new"] = paid_summary["spend"] / paid_summary["new_customers"]
paid_summary = paid_summary.sort_values("iroas", ascending=False)

paid_display = pd.DataFrame({
    "Spend": paid_summary["spend"].map("${:,.0f}".format),
    "Attributed revenue": paid_summary["attributed"].map("${:,.0f}".format),
    "ROAS": paid_summary["roas"].map("{:.1f}x".format),
    "Est. iROAS": paid_summary["iroas"].map("{:.2f}x".format),
    "New customers": paid_summary["new_customers"].astype(int),
    "CAC (new)": paid_summary["cac_new"].map("${:,.0f}".format),
})
display(paid_display)
print(f"Site incrementality factor: {incrementality_factor:.2f} "
      f"(${incremental_net/1e3:,.1f}K incremental / ${paid_summary['attributed'].sum()/1e3:,.1f}K attributed)")

social = paid_summary.loc["Paid Social"]
print(f"Paid Social: {social['new_customers']:.0f} of {len(new_ids)} new customers "
      f"({social['new_customers']/len(new_ids):.0%}) at ${social['cac_new']:,.0f} CAC vs "
      f"${early_value.mean():,.0f} early value — pays back as acquisition, thinly, not as revenue.")

organic_direct = active_channels[active_channels["channel"].isin(["Organic", "Direct"])]
od_pre = channels[(channels["campaign_phase"] == "Pre-campaign")
                  & (channels["channel"].isin(["Organic", "Direct"]))]
print(f"Unpaid spillover: Organic + Direct attributed revenue/day "
      f"${od_pre['attributed_revenue'].sum()/92:,.0f} pre -> "
      f"${organic_direct['attributed_revenue'].sum()/ACTIVE_DAYS:,.0f} active "
      f"({(organic_direct['attributed_revenue'].sum()/ACTIVE_DAYS)/(od_pre['attributed_revenue'].sum()/92)-1:+.0%})")

,Spend,Attributed revenue,ROAS,Est. iROAS,New customers,CAC (new)
channel,,,,,,
Email,"$3,255","$83,945",25.8x,8.65x,39,$83
SMS,"$2,490","$24,017",9.6x,3.24x,20,$124
Affiliate,"$4,991","$20,771",4.2x,1.40x,61,$82
Paid Search,"$18,849","$58,008",3.1x,1.03x,175,$108
Paid Social,"$44,969","$44,247",1.0x,0.33x,297,$151


Site incrementality factor: 0.34 ($77.5K incremental / $231.0K attributed)
Paid Social: 297 of 730 new customers (41%) at $151 CAC vs $188 early value — pays back as acquisition, thinly, not as revenue.
Unpaid spillover: Organic + Direct attributed revenue/day $4,083 pre -> $5,701 active (+40%)


In [25]:
iroas_colors = [SALOMON_BLUE if value >= 1 else BLUE_LIGHT for value in paid_summary["iroas"]]

fig = go.Figure(go.Bar(
    x=paid_summary["iroas"],
    y=paid_summary.index,
    orientation="h",
    marker=dict(color=iroas_colors, line=dict(color=SALOMON_BLUE, width=0.9)),
    text=paid_summary["iroas"].map("{:.2f}×".format),
    textposition="outside",
    textfont=dict(color=INK, size=12),
    cliponaxis=False,
    customdata=paid_summary[["spend", "attributed", "cac_new"]],
    hovertemplate=(
        "%{y}<br>Estimated iROAS: %{x:.2f}×"
        "<br>Spend: $%{customdata[0]:,.0f}"
        "<br>Attributed revenue: $%{customdata[1]:,.0f}"
        "<br>New-customer CAC: $%{customdata[2]:,.0f}<extra></extra>"
    ),
))
fig.add_vline(x=1, line=dict(color=MUTED, width=1.5, dash="dash"))
fig.add_annotation(
    x=1, y=0.96, xref="x", yref="paper", text="1.0× BREAKEVEN",
    showarrow=False, xanchor="left", yanchor="top",
    bgcolor="rgba(255,255,255,0.82)", borderpad=3, font=dict(size=10, color=MUTED),
)
fig.update_xaxes(
    title=f"Estimated incremental ROAS · attributed ROAS × {incrementality_factor:.2f} site factor",
    ticksuffix="×",
    range=[0, paid_summary["iroas"].max() * 1.16],
)
fig.update_yaxes(autorange="reversed", title=None, showgrid=False)
show_campaign_chart(
    fig,
    "Estimated incremental ROAS by paid channel",
    height=390,
    left_margin=112,
    export_name="04_incremental_roas_by_channel.png",
)

/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/04_incremental_roas_by_channel.png


Deck version -- breakeven-or-better channels in blue, below breakeven in gray.

In [26]:
# Deck-formatted iROAS by channel for the readout slide.
iroas_deck_colors = [SALOMON_BLUE if v >= 1 else MUTED for v in paid_summary["iroas"]]

fig = go.Figure(go.Bar(
    x=paid_summary["iroas"],
    y=paid_summary.index,
    orientation="h",
    marker=dict(color=iroas_deck_colors),
    text=[f"<b>{v:.1f}\u00d7</b>" for v in paid_summary["iroas"]],
    textposition="outside",
    textfont=dict(color=INK, size=13),
    cliponaxis=False,
    customdata=paid_summary[["roas", "spend", "cac_new"]],
    hovertemplate=("%{y}<br>Estimated iROAS: %{x:.2f}\u00d7"
                   "<br>Attributed ROAS: %{customdata[0]:.1f}\u00d7"
                   "<br>Spend: $%{customdata[1]:,.0f}"
                   "<br>New-customer CAC: $%{customdata[2]:,.0f}<extra></extra>"),
))
fig.add_vline(x=1, line=dict(color=MUTED, width=1.3, dash="dash"))
fig.add_annotation(x=1, y=1.0, yref="paper", yshift=6, showarrow=False,
                   text="BREAKEVEN 1.0\u00d7", font=dict(size=10, color=MUTED))
fig.update_yaxes(autorange="reversed", showgrid=False, title=None,
                 tickfont=dict(color=INK, size=13))
fig.update_xaxes(visible=False, showgrid=False, range=[0, paid_summary["iroas"].max() * 1.14])
show_campaign_chart(
    fig, "Incremental ROAS by channel",
    height=470, width=860, left_margin=110,
    export_name="13_channel_iroas_deck.png",
)


Saved plots/13_channel_iroas_deck.png


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Same channel numbers as a deck table. spend share, attributed-revenue share, attributed ROAS, incremental ROAS side by side.

In [27]:
# Deck-formatted channel summary table for the readout slide.
by_spend = paid_summary.sort_values("spend", ascending=False)
tot_spend = by_spend["spend"].sum()
tot_attr = by_spend["attributed"].sum()
tot_new = by_spend["new_customers"].sum()

def money(v): return f"${v:,.0f}"
def pct(v): return f"{v:.0%}"
def mult(v): return f"{v:.1f}\u00d7"
def count(v): return f"{int(v):,}"

header = ["Channel", "Spend", "% of spend", "Attributed revenue", "% of attributed", "ROAS", "Est. iROAS",
          "New customers", "CAC (new)"]
rows = [[ch, money(r["spend"]), pct(r["spend"]/tot_spend), money(r["attributed"]),
         pct(r["attributed"]/tot_attr), mult(r["roas"]), mult(r["iroas"]),
         count(r["new_customers"]), money(r["cac_new"])]
        for ch, r in by_spend.iterrows()]
rows.append([f"<b>{v}</b>" for v in
             ["Total", money(tot_spend), "100%", money(tot_attr), "100%",
              mult(tot_attr/tot_spend), mult(tot_attr/tot_spend * incrementality_factor),
              count(tot_new), money(tot_spend/tot_new)]])
columns = list(zip(*rows))

row_fills = [[SURFACE if i % 2 else WHITE for i in range(len(by_spend))] + [BLUE_LIGHT]]
fig = go.Figure(go.Table(
    columnwidth=[1.5, 1, 1, 1.4, 1.2, 0.8, 1, 1.1, 1],
    header=dict(
        values=[f"<b>{h}</b>" for h in header],
        fill_color=SALOMON_BLUE, font=dict(color=WHITE, size=13), align=["left"] + ["right"]*8,
        height=34, line_color=SALOMON_BLUE,
    ),
    cells=dict(
        values=columns,
        fill_color=row_fills * len(header),
        font=dict(color=INK, size=13), align=["left"] + ["right"]*8,
        height=32, line_color=GRID,
    ),
))
show_campaign_chart(
    fig, "Channel Breakdown",
    height=400, width=1180,
    export_name="14_channel_summary_table.png",
)

/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/14_channel_summary_table.png


DECK TABLE above

### Brand awareness -- branded search

Splitting branded queries ("salomon", "vestal pro") from generic category terms ("trail running shoes"). Branded search is the best awareness signal without a survey. only moves when people already have us in mind.

Data's from `src/generate_search_console_data.py`, same Mar 1 to Jul 20 window as everything else.

Real question: did this move brand demand durably, or just spike while media was live and fall back after?

In [28]:
search = pd.read_csv(DATA_DIR / "search_console_daily.csv", parse_dates=["date"])
assert (search["campaign_phase"].unique() == ["Pre-campaign", "Active campaign", "Post-campaign"]).all()

# Brand impressions run ~9x below non-brand (high brand CTR), so the impressions
# chart is indexed to the pre-campaign average; clicks are shown in absolute terms.
pre_mask = search["campaign_phase"] == "Pre-campaign"
for col in ["brand_impressions", "nonbrand_impressions"]:
    search[f"{col}_idx"] = (search[col] / search.loc[pre_mask, col].mean() * 100).rolling(7, min_periods=1).mean()
for seg in ["brand", "nonbrand"]:
    search[f"{seg}_ctr"] = (search[f"{seg}_clicks"] / search[f"{seg}_impressions"]).rolling(7, min_periods=1).mean()

for metric, ylab, title, export_name, hover_fmt, chart_width in [
    ("clicks", "Search clicks per day", "Branded vs non-brand search clicks",
     "15_brand_search_clicks.png", ",.0f", 1000),
    ("impressions_idx", "Impressions (pre-campaign = 100)",
     "Branded vs non-brand search impressions, scaled to 100 \u00b7 7-day average",
     "16_brand_search_impressions.png", ",.0f", 1250),
]:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=search["date"], y=search[f"brand_{metric}"],
        name="Brand", mode="lines", line=dict(color=SALOMON_BLUE, width=2.4),
        hovertemplate="%{x|%b %d}<br>Brand: %{y:" + hover_fmt + "}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=search["date"], y=search[f"nonbrand_{metric}"],
        name="Non-brand", mode="lines", line=dict(color="#000000", width=2),
        hovertemplate="%{x|%b %d}<br>Non-brand: %{y:" + hover_fmt + "}<extra></extra>",
    ))
    if metric == "impressions_idx":
        fig.add_hline(y=100, line=dict(color=GRID, width=1.2, dash="dash"))
        fig.add_trace(go.Scatter(
            x=search["date"], y=search["brand_ctr"],
            name="Brand CTR", mode="lines", yaxis="y2",
            line=dict(color=SALOMON_BLUE, width=1.6, dash="dash"),
            hovertemplate="%{x|%b %d}<br>Brand CTR: %{y:.1%}<extra></extra>",
        ))
        fig.add_trace(go.Scatter(
            x=search["date"], y=search["nonbrand_ctr"],
            name="Non-brand CTR", mode="lines", yaxis="y2",
            line=dict(color="#000000", width=1.6, dash="dash"),
            hovertemplate="%{x|%b %d}<br>Non-brand CTR: %{y:.1%}<extra></extra>",
        ))
        fig.update_layout(yaxis2=dict(
            title=dict(text="CTR (7-day average)", font=dict(color=MUTED)),
            overlaying="y", side="right", tickformat=".0%",
            range=[0, 0.62], showgrid=False, tickfont=dict(color=MUTED), automargin=True,
        ))
    fig.add_vline(x=ACT_START, line=dict(color=MUTED, width=1))
    fig.add_vline(x=ACT_END, line=dict(color=MUTED, width=1))
    fig.add_annotation(
        x=ACT_START + (ACT_END - ACT_START) / 2, y=0.97, xref="x", yref="paper",
        text="CAMPAIGN", showarrow=False, yanchor="top",
        bgcolor="rgba(255,255,255,0.82)", borderpad=3, font=dict(size=10, color=MUTED),
    )
    fig.update_layout(legend=dict(
        orientation="h", x=0.01, y=1.04, xanchor="left", yanchor="bottom",
        font=dict(color=INK),
    ))
    fig.update_xaxes(title=None)
    fig.update_layout(yaxis=dict(title=ylab, tickformat="~s", rangemode="tozero"))
    show_campaign_chart(
        fig, title,
        height=440, width=chart_width, hovermode="x unified",
        export_name=export_name,
    )

phase_means = search.groupby("campaign_phase")[
    ["brand_clicks", "brand_impressions", "nonbrand_clicks", "nonbrand_impressions"]].mean()
pre, act, post = (phase_means.loc[p] for p in ["Pre-campaign", "Active campaign", "Post-campaign"])
final_two_weeks = search[search["date"] > search["date"].max() - pd.Timedelta(days=14)]
brand_share = search.groupby("campaign_phase").apply(
    lambda g: g["brand_clicks"].sum() / (g["brand_clicks"] + g["nonbrand_clicks"]).sum(),
    include_groups=False)

print(f"Brand clicks/day: {pre['brand_clicks']:,.0f} pre -> {act['brand_clicks']:,.0f} campaign "
      f"({act['brand_clicks']/pre['brand_clicks']-1:+.0%}) -> {post['brand_clicks']:,.0f} post "
      f"({post['brand_clicks']/pre['brand_clicks']-1:+.0%})")
print(f"Brand impressions/day: {pre['brand_impressions']:,.0f} pre -> {act['brand_impressions']:,.0f} campaign "
      f"({act['brand_impressions']/pre['brand_impressions']-1:+.0%}) -> {post['brand_impressions']:,.0f} post "
      f"({post['brand_impressions']/pre['brand_impressions']-1:+.0%})")
print(f"Non-brand clicks/day: {pre['nonbrand_clicks']:,.0f} pre -> {act['nonbrand_clicks']:,.0f} campaign "
      f"({act['nonbrand_clicks']/pre['nonbrand_clicks']-1:+.0%}) -> {post['nonbrand_clicks']:,.0f} post "
      f"({post['nonbrand_clicks']/pre['nonbrand_clicks']-1:+.0%})")
print(f"Brand share of category clicks: {brand_share['Pre-campaign']:.0%} pre -> "
      f"{brand_share['Post-campaign']:.0%} post")
print(f"Final two weeks (Jul 7-20), a month after the campaign ended: "
      f"{final_two_weeks['brand_clicks'].mean():,.0f} brand clicks/day, still "
      f"{final_two_weeks['brand_clicks'].mean()/pre['brand_clicks']-1:+.0%} vs pre-campaign \u2014 "
      f"the lift has held rather than decayed with the media.")


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/15_brand_search_clicks.png


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/16_brand_search_impressions.png


Brand clicks/day: 839 pre -> 1,010 campaign (+20%) -> 1,021 post (+22%)
Brand impressions/day: 2,901 pre -> 3,440 campaign (+19%) -> 3,531 post (+22%)
Non-brand clicks/day: 1,501 pre -> 1,619 campaign (+8%) -> 1,577 post (+5%)
Brand share of category clicks: 36% pre -> 39% post
Final two weeks (Jul 7-20), a month after the campaign ended: 1,026 brand clicks/day, still +22% vs pre-campaign — the lift has held rather than decayed with the media.


### Inventory, returns, service

Last section. checking if we left sales on the table from stockouts, and whether returns/CS contacts got weird.

In [29]:
vestal_flight = vestal[vestal["date"] >= ACT_START].copy()
vestal_flight["week"] = (vestal_flight["date"] - ACT_START).dt.days // 7 + 1
active_weeks = vestal_flight[vestal_flight["campaign_phase"] == "Active campaign"]
weekly = active_weeks.groupby("week").agg(
    availability=("core_size_availability", "mean"),
    sessions=("sessions", "sum"), units=("units_sold", "sum"))
weekly["sell_through"] = weekly["units"] / weekly["sessions"]
weekly_display = weekly.copy()
weekly_display["availability"] = weekly_display["availability"].map("{:.1%}".format)
weekly_display["sell_through"] = weekly_display["sell_through"].map("{:.2%}".format)
display(weekly_display)

week1_conv = weekly.loc[1, "sell_through"]
later = weekly.loc[2:]
unmet_units = week1_conv * later["sessions"].sum() - later["units"].sum()
asp = vestal_direct / vestal_units
print(f"If weeks 2–3 had matched the week-1 conversion rate, sales would have been about {unmet_units:.0f} units higher "
      f"(~${unmet_units*asp/1e3:,.1f}K). Conversion remained steady during the campaign.")

post_availability = vestal[vestal["campaign_phase"] == "Post-campaign"]["core_size_availability"].mean()
post_revenue = vestal[vestal["campaign_phase"] == "Post-campaign"]["net_revenue"].sum()
print(f"Core-size availability averaged {post_availability:.0%} after the campaign, "
      f"compared with {weekly['availability'].iloc[-1]:.0%} in week 3. The stockout happened later and "
      f"limited the ${post_revenue/1e3:,.1f}K in post-campaign sales.")

,availability,sessions,units,sell_through
week,,,,
1,98.0%,5795,268,4.62%
2,91.9%,5139,246,4.79%
3,73.7%,4174,180,4.31%


If weeks 2–3 had matched the week-1 conversion rate, sales would have been about 5 units higher (~$0.7K). Conversion remained steady during the campaign.
Core-size availability averaged 44% after the campaign, compared with 74% in week 3. The stockout happened later and limited the $56.5K in post-campaign sales.


In [30]:
availability_window = vestal[vestal["date"] >= ACT_START]

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.64, 0.36],
    horizontal_spacing=0.13,
)
fig.add_trace(go.Scatter(
    x=availability_window["date"],
    y=availability_window["core_size_availability"],
    mode="lines",
    line=dict(color=SALOMON_BLUE, width=2.6),
    fill="tozeroy",
    fillcolor="rgba(191, 213, 237, 0.35)",
    hovertemplate="%{x|%b %d}<br>Availability: %{y:.1%}<extra></extra>",
    showlegend=False,
), row=1, col=1)
fig.add_hline(y=0.9, row=1, col=1, line=dict(color=MUTED, width=1.3, dash="dash"))
fig.add_vline(x=ACT_END, row=1, col=1, line=dict(color=MUTED, width=1))
fig.add_annotation(
    x=availability_window["date"].iloc[2], y=0.91, xref="x", yref="y",
    text="90% HEALTHY", showarrow=False, xanchor="left",
    font=dict(size=10, color=MUTED),
)
fig.add_annotation(
    x=ACT_END, y=0.54, xref="x", yref="y",
    text="CAMPAIGN ENDS", showarrow=False, xanchor="left", textangle=-90,
    font=dict(size=10, color=MUTED),
)

fig.add_trace(go.Bar(
    x=weekly.index.astype(str),
    y=weekly["sell_through"],
    marker=dict(color=SALOMON_BLUE, line=dict(color=INK, width=0.7)),
    text=weekly["sell_through"].map("{:.1%}".format),
    textposition="outside",
    textfont=dict(color=INK),
    cliponaxis=False,
    hovertemplate="Week %{x}<br>Sell-through: %{y:.2%}<extra></extra>",
    showlegend=False,
), row=1, col=2)

fig.update_xaxes(title=None, row=1, col=1)
fig.update_yaxes(title="Core-size availability", tickformat=".0%", range=[0, 1.04], row=1, col=1)
fig.update_xaxes(title="Campaign week", type="category", row=1, col=2)
fig.update_yaxes(title="Units per Vestal Pro session", tickformat=".1%", rangemode="tozero", row=1, col=2)
show_campaign_chart(
    fig,
    "Core sizes lasted through the campaign, then ran low",
    height=440,
    export_name="05_inventory_availability_and_sell_through.png",
)

Saved plots/05_inventory_availability_and_sell_through.png


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Deck version of the supply issue. weekly core-size availability vs overall in-stock rate, plus unmet demand (holding the campaign conversion rate constant into the post-campaign weeks). weeks count from the June 1 launch, dropped week 8 since it's only partial.

In [31]:
# Weekly availability and unmet demand for the inventory slide.
vp_flight = products[(products["franchise"] == "Vestal Pro") & (products["date"] >= ACT_START)].copy()
vp_flight["week"] = (vp_flight["date"] - ACT_START).dt.days // 7 + 1
weekly_supply = vp_flight[vp_flight["week"] <= 7].groupby("week").agg(
    core_avail=("core_size_availability", "mean"),
    in_stock=("in_stock_rate", "mean"),
    sessions=("sessions", "sum"),
    units=("units_sold", "sum"),
    revenue=("net_revenue", "sum"),
)
weekly_supply["sell_through"] = weekly_supply["units"] / weekly_supply["sessions"]

week_labels = [f"Wk {w}" for w in weekly_supply.index]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=week_labels, y=weekly_supply["in_stock"],
    name="Overall in-stock rate", mode="lines",
    line=dict(color=MUTED, width=2.2),
    hovertemplate="%{x}<br>Overall in-stock: %{y:.0%}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=week_labels, y=weekly_supply["core_avail"],
    name="Core-size availability", mode="lines+markers+text",
    line=dict(color=SALOMON_BLUE, width=2.8), marker=dict(size=7),
    text=[f"{v:.0%}" for v in weekly_supply["core_avail"]],
    textposition="bottom center", textfont=dict(color=SALOMON_BLUE, size=12),
    cliponaxis=False,
    hovertemplate="%{x}<br>Core sizes: %{y:.0%}<extra></extra>",
))
fig.add_hline(y=0.9, line=dict(color=SALOMON_GREEN, width=1.3, dash="dash"))
fig.add_annotation(x=0.02, y=0.905, xref="paper", yref="y", text="90% HEALTHY",
                   showarrow=False, xanchor="left", yanchor="bottom",
                   font=dict(size=10, color=SALOMON_GREEN))
fig.add_vline(x=2.5, line=dict(color=MUTED, width=1))
fig.add_annotation(x=2.5, y=0.99, xref="x", yref="y", text="CAMPAIGN ENDS JUN 20",
                   showarrow=False, xanchor="left", font=dict(size=10, color=MUTED))
fig.update_layout(legend=dict(orientation="h", x=0.01, y=-0.12, xanchor="left", font=dict(color=INK)))
fig.update_xaxes(title=None, showgrid=False, tickfont=dict(color=INK, size=13))
fig.update_yaxes(title=None, tickformat=".0%", range=[0.38, 1.04])
show_campaign_chart(
    fig, "Vestal Pro availability by week",
    height=470, width=900,
    export_name="17_availability_by_week.png",
)

# Unmet demand: campaign-period conversion applied to post-campaign traffic.
campaign_st = weekly_supply.loc[1:3, "units"].sum() / weekly_supply.loc[1:3, "sessions"].sum()
asp_flight = weekly_supply["revenue"].sum() / weekly_supply["units"].sum()
weekly_supply["modeled_units"] = campaign_st * weekly_supply["sessions"]
late_weeks = weekly_supply.loc[4:]
unmet_units = late_weeks["modeled_units"].sum() - late_weeks["units"].sum()
unmet_revenue = unmet_units * asp_flight

daily_supply = vp_flight.sort_values("date").copy()
daily_supply["conv_7d"] = (daily_supply["units_sold"] / daily_supply["sessions"]).rolling(7, min_periods=1).mean()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, row_heights=[0.30, 0.70],
    vertical_spacing=0.09, specs=[[{}], [{"secondary_y": True}]],
)
fig.add_trace(go.Scatter(
    x=daily_supply["date"], y=daily_supply["sessions"],
    name="PDP views", mode="lines",
    line=dict(color=BLUE_BRIGHT, width=2.2),
    fill="tozeroy", fillcolor="rgba(191, 213, 237, 0.30)",
    hovertemplate="%{x|%b %d}<br>PDP views: %{y:,.0f}<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=daily_supply["date"], y=daily_supply["core_size_availability"],
    name="Core-size availability", mode="lines",
    line=dict(color=SALOMON_BLUE, width=2.8),
    hovertemplate="%{x|%b %d}<br>Core sizes: %{y:.0%}<extra></extra>",
), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(
    x=daily_supply["date"], y=daily_supply["conv_7d"],
    name="PDP conversion (units/view, 7-day average)", mode="lines",
    line=dict(color="#000000", width=2.2, dash="dash"),
    hovertemplate="%{x|%b %d}<br>Conversion (7d avg): %{y:.1%}<extra></extra>",
), row=2, col=1, secondary_y=True)
fig.add_hline(y=0.9, row=2, col=1, secondary_y=False,
              line=dict(color=SALOMON_GREEN, width=1.3, dash="dash"))
fig.add_annotation(x=daily_supply["date"].iloc[1], y=0.905, xref="x2", yref="y2",
                   text="90% HEALTHY", showarrow=False, xanchor="left", yanchor="bottom",
                   font=dict(size=10, color=SALOMON_GREEN))
for row in (1, 2):
    fig.add_vline(x=ACT_END, row=row, col=1, line=dict(color=MUTED, width=1))
fig.add_annotation(x=ACT_END, y=0.99, xref="x2", yref="y2", text="CAMPAIGN ENDS JUN 20",
                   showarrow=False, xanchor="left", font=dict(size=10, color=MUTED))
fig.add_annotation(
    x=pd.Timestamp("2026-07-06"), y=0.885, xref="x2", yref="y2",
    text=(f"<b>Conversion gap after wk 3 \u2248{unmet_units:.0f} units \u00b7 "
          f"\u2248${unmet_revenue/1e3:,.1f}K</b>"),
    showarrow=False, font=dict(size=12, color=INK),
    bgcolor="rgba(255,255,255,0.85)", borderpad=3,
)
fig.update_layout(legend=dict(orientation="h", x=0.01, y=-0.10, xanchor="left", font=dict(color=INK)),
                  )
fig.update_yaxes(title=None, tickformat="~s", rangemode="tozero", row=1, col=1)
fig.update_yaxes(title="Core-size availability", tickformat=".0%", range=[0.35, 1.03],
                 row=2, col=1, secondary_y=False)
fig.update_yaxes(title=dict(text="PDP conversion", font=dict(color=MUTED)), tickformat=".0%",
                 range=[0.02, 0.052], showgrid=False, tickfont=dict(color=MUTED),
                 row=2, col=1, secondary_y=True)
fig.update_xaxes(showgrid=False, tickfont=dict(color=INK, size=12))
show_campaign_chart(
    fig, "PDP conversion tracked size availability",
    height=560, width=980,
    export_name="18_pdp_conversion_availability.png",
)

pre_conv = ecommerce[ecommerce["campaign_phase"] == "Pre-campaign"]["conversion_rate"].mean()
ec_flight = ecommerce[ecommerce["date"] >= ACT_START].copy()
ec_flight["week"] = (ec_flight["date"] - ACT_START).dt.days // 7 + 1
site_conv = ec_flight[ec_flight["week"] <= 7].groupby("week")["conversion_rate"].mean()

print(f"Core-size availability: {weekly_supply.loc[1,'core_avail']:.0%} wk 1 -> "
      f"{weekly_supply.loc[3,'core_avail']:.0%} wk 3 -> {weekly_supply.loc[7,'core_avail']:.0%} wk 7; "
      f"overall in-stock {weekly_supply.loc[7,'in_stock']:.0%} by wk 7")
print(f"Vestal Pro sell-through: {weekly_supply.loc[1:3,'sell_through'].mean():.1%} campaign avg -> "
      f"{weekly_supply.loc[7,'sell_through']:.1%} wk 7 "
      f"({weekly_supply.loc[7,'sell_through']/campaign_st-1:+.0%})")
print(f"Unmet demand weeks 4-7: \u2248{unmet_units:.0f} units, \u2248${unmet_revenue/1e3:,.1f}K "
      f"at the ${asp_flight:,.0f} average selling price")
print(f"Site conversion: {site_conv.loc[1]:.2%} wk 1 -> {site_conv.loc[7]:.2%} wk 7, "
      f"vs {pre_conv:.2%} pre-campaign average")
speed = products[products["franchise"] == "Speedcross"]
speed_shift = (speed[speed["campaign_phase"] == "Post-campaign"]["units_sold"].mean()
               / speed[speed["campaign_phase"] == "Pre-campaign"]["units_sold"].mean() - 1)
print(f"Speedcross post-campaign units are {speed_shift:+.0%} vs pre \u2014 shoppers largely did not "
      f"substitute, so the unmet demand reads as lost, not shifted.")


Saved plots/17_availability_by_week.png


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


/tmp/ipykernel_180492/6123294.py:94: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(


Saved plots/18_pdp_conversion_availability.png


Core-size availability: 98% wk 1 -> 71% wk 3 -> 43% wk 7; overall in-stock 62% by wk 7
Vestal Pro sell-through: 4.6% campaign avg -> 3.0% wk 7 (-35%)
Unmet demand weeks 4-7: ≈112 units, ≈$16.2K at the $145 average selling price
Site conversion: 2.52% wk 1 -> 2.31% wk 7, vs 2.33% pre-campaign average
Speedcross post-campaign units are +1% vs pre — shoppers largely did not substitute, so the unmet demand reads as lost, not shifted.


In [32]:
pre_ec = ecommerce[ecommerce["campaign_phase"] == "Pre-campaign"]
act_ec = ecommerce[ecommerce["campaign_phase"] == "Active campaign"]
post_ec = ecommerce[ecommerce["campaign_phase"] == "Post-campaign"]
vestal_post_orders = orders[(orders["campaign_phase"] == "Post-campaign")
                          & (orders["primary_product"] == "Vestal Pro")]
contacts_per_order = (services.groupby("campaign_phase").size()
                      / orders.groupby("campaign_phase").size())
reason_mix = pd.DataFrame({
    "pre_per_day": services[services["campaign_phase"] == "Pre-campaign"].groupby("contact_reason").size() / 92,
    "active_per_day": services[services["campaign_phase"] == "Active campaign"].groupby("contact_reason").size() / ACTIVE_DAYS,
}).fillna(0)
reason_mix["ratio"] = reason_mix["active_per_day"] / reason_mix["pre_per_day"].replace(0, np.nan)

checks = pd.DataFrame({
    "Metric": ["Average order value", "Site conversion rate", "Site return rate",
                  "Vestal Pro return rate (orders)", "Other-footwear return rate (orders)",
                  "Service contacts per order"],
    "Pre": [f"${pre_ec['net_revenue'].sum()/pre_ec['orders'].sum():,.0f}",
            f"{pre_ec['orders'].sum()/pre_ec['sessions'].sum():.2%}",
            f"{pre_ec['return_rate'].mean():.1%}", "—",
            f"{orders[(orders['campaign_phase']=='Pre-campaign') & (orders['primary_product'].isin(EXISTING_FRANCHISES))]['returned'].mean():.1%}",
            f"{contacts_per_order['Pre-campaign']:.1%}"],
    "Active": [f"${act_ec['net_revenue'].sum()/act_ec['orders'].sum():,.0f}",
               f"{act_ec['orders'].sum()/act_ec['sessions'].sum():.2%}",
               f"{act_ec['return_rate'].mean():.1%}",
               f"{vestal_orders['returned'].mean():.1%}",
               f"{other_fw['returned'].mean():.1%}",
               f"{contacts_per_order['Active campaign']:.1%}"],
    "Post": ["—",
             f"{post_ec['orders'].sum()/post_ec['sessions'].sum():.2%}",
             f"{post_ec['return_rate'].mean():.1%}",
             f"{vestal_post_orders['returned'].mean():.1%}", "—",
             f"{contacts_per_order['Post-campaign']:.1%}"],
})
display(checks)

top_risers = reason_mix.sort_values("ratio", ascending=False).head(3)
print("Fastest-rising contact reasons (active vs pre, per day):")
for reason, row in top_risers.iterrows():
    print(f"  {reason}: {row['pre_per_day']:.1f}/day -> {row['active_per_day']:.1f}/day ({row['ratio']:.1f}x)")
print("Product-defect contacts stayed near zero — the return story is sizing, not quality.")

,Metric,Pre,Active,Post
0,Average order value,$124,$133,—
1,Site conversion rate,2.33%,2.47%,2.31%
2,Site return rate,7.0%,8.4%,8.0%
3,Vestal Pro return rate (orders),—,15.0%,14.1%
4,Other-footwear return rate (orders),8.6%,8.5%,—
5,Service contacts per order,4.4%,7.0%,6.7%


Fastest-rising contact reasons (active vs pre, per day):
  Product comparison: 0.5/day -> 2.4/day (4.6x)
  Inventory availability: 0.4/day -> 1.3/day (3.3x)
  Sizing and fit: 0.9/day -> 2.0/day (2.3x)
Product-defect contacts stayed near zero — the return story is sizing, not quality.


## Conclusions and next steps

Revenue:
- footwear revenue **+35.8%** vs baseline (**+$69.3K**), 95% CI +32% to +40%
- total net revenue **+33.1%** vs baseline (**+$77.5K**) = **1.80×** the incremental media spend
- after the extra **$43.0K** in media, ~**$34.5K** left over
- no COGS so can't get to an actual profit number

Two ways of counting revenue line up within rounding: **$99.8K** direct - **$30.5K** cannibalized + **$8.2K** halo ≈ **$77.5K** site-level number.

New customers:
- **40.2%** of buyers new to Salomon, well above the 25% goal
- CAC **$102** vs **$188** in early value already
- **28.1%** repeat purchase rate post-campaign
- email/SMS daily sign-ups up **35%** / **37%**

Cannibalization -- the one I keep coming back to:
- 3 estimates: **19% to 32%** of Vestal Pro units, median ~**28%**
- goal is under 10%, all three estimates blow past it
- Ultra Glide hit hardest, then Genesis
- still net positive overall but this isn't "safely below" the limit. more like well over it depending which baseline you trust

Apparel/accessories -- halo worked:
- **+20.3%** vs baseline, 95% CI +14% to +27%
- attach rate **21.6%** on Vestal Pro orders vs **17.5%** on other footwear orders
- ~**$9.3K** attached to Vestal Pro orders specifically, standalone demand flat
- keep doing the bundle/cross-sell placements

Channels:
- iROAS: **8.7× email, 3.2× SMS, 1.4× affiliate**
- paid search ~breakeven
- paid social **0.33×**, rough on iROAS but bringing in **41%** of new customers at **$151 CAC** vs **$188** early value. don't kill it on iROAS alone
- want a real CAC target + day-90 value read next time
- reminder: all channel numbers use the same 0.34 incrementality factor. not a real channel experiment

Inventory:
- fine during the actual campaign -- low point **74%** availability week 3, ~**$1K** unmet demand
- fell off after -- ~**44%** availability post-campaign, capping a **$56.5K** sales tail still **17%** above pre-campaign daily rate
- initial buy was fine, replenishment after launch wasn't

Next time:
- real CRM holdout + geo test
- get COGS and unsubscribe data flowing
- set a CAC target before launch not after
- site return rate **8.4%** during campaign, Vestal Pro ~**15%**, mostly sizing
- returns still maturing, check again once post-campaign period closes out